#### Note: This colab only contains short discussion/highlights of task done. Please visit the google document for my detailed writeup:
Google document link: https://docs.google.com/document/d/1_bWr3cJCFDDScIspYEVpgFat8taDgVD9NEM2zIkvgsY/edit?usp=sharing

### TASK 1: Understanding MaxText Data format

To complete task 1 I explored files inside folder `maxtext/input_pipeline` , especially `input_pipeline_inferace.py`.

Here are the dataset formats supported by MaxText:

- `synthetic dataset`: fake dataset (token ids) batches are created in memory, rather than loading any dataset. Used for bechmarkiing hardware throughput purposes etc.

- `grain`: you can load the dataset using following file types:    
  - `arrayrecord`: ArrayRecord file format supports parallel read, write, and random access by record index.

  - `parquet`: Apache parquet format - open-source columnar storage format.


  - `tfrecord`: TensorFlow's standard format for storing a sequence of binary records.




### TASK 2:



To complete task 2 let's first look at the resources available.

In [1]:

import os, psutil, platform

print("=== Hardware Info ===")
print(f"CPU cores: {os.cpu_count()}")
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")
print(f"Platform: {platform.platform()}")

# Check JAX backend
import jax
print(f"\nJAX version: {jax.__version__}")
print(f"JAX backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

=== Hardware Info ===
CPU cores: 24
Total RAM: 47.05 GB
Available RAM: 45.03 GB
Platform: Linux-6.6.122+-x86_64-with-glibc2.35


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(



JAX version: 0.7.2
JAX backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


Let's answer the most important question: how much memory is occupied when we train the model and what actually require memory ?

Training requires forward and backward pass:
this is what consume memory:
- Model parameters
- gradients
- optimizer states
- also one thing to note: activations stored from forward pass for backward pass also requires memory.

What also matters is if your model parameter's precision format is in FP32 (32 bit), FP16 (16 bits) or is it quantized (8 bit, 4bit)

For optimizer states, usually it is in fp32

In [2]:
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext


Cloning into 'maxtext'...
remote: Enumerating objects: 97119, done.
remote: Counting objects: 100% (2295/2295), done.
remote: Compressing objects: 100% (1161/1161), done.
remote: Total 97119 (delta 1876), reused 1148 (delta 1134), pack-reused 94824 (from 5)
Receiving objects: 100% (97119/97119), 411.67 MiB | 42.71 MiB/s, done.
Resolving deltas: 100% (70821/70821), done.
/content/maxtext


In [3]:
!uv venv --python 3.12 --seed exp
!source exp/bin/activate

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment with seed packages at: exp
 + pip==26.1.2
Activate with: source exp/bin/activate


In [4]:
!pip install -r src/dependencies/requirements/generated_requirements/tpu-requirements.txt
!pip install -e .


Ignoring nest-asyncio: markers 'sys_platform == "win32"' don't match your environment
Ignoring tzdata: markers 'sys_platform == "emscripten" or sys_platform == "win32"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of gcsfs to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━

Obtaining file:///content/maxtext
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for maxtext (pyproject.toml) ... done
  Created wheel for maxtext: filename=maxtext-0.2.3-py3-none-any.whl size=19412 sha256=4f76513a610e53efcba7d508aac5812bc1c89366a7ccae1c2ef654d71bcbefb3
  Stored in directory: /tmp/pip-ephem-wheel-cache-b2t89j_p/wheels/6c/d1/14/ccb96d070c3f58903f006ad2bc5a7b92d25df031b495676b7b
Successfully built maxtext


In [5]:
!pip uninstall -y torch torchvision torchaudio torch_xla
!pip install torch --index-url https://download.pytorch.org/whl/cpu

Found existing installation: torch 2.9.0+cpu
Uninstalling torch-2.9.0+cpu:
  Successfully uninstalled torch-2.9.0+cpu
Found existing installation: torchvision 0.24.0+cpu
Uninstalling torchvision-0.24.0+cpu:
  Successfully uninstalled torchvision-0.24.0+cpu
Found existing installation: torchaudio 2.9.0+cpu
Uninstalling torchaudio-2.9.0+cpu:
  Successfully uninstalled torchaudio-2.9.0+cpu
Found existing installation: torch-xla 2.9.0
Uninstalling torch-xla-2.9.0:
  Successfully uninstalled torch-xla-2.9.0
Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.3/192.3 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 930.8/930.8 kB 28.7 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.1
    Uninstalling setuptools-82.0.1:
      Successfully uninstalled setuptools-82.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

In [5]:
# ## Using attention=dot_product (by default it is autoselected which might choose flash attention). flash attention kernels probably
# ## might be written in triton and Triton native GPU kernel execution failed on Turing architecture (T4) and requires ampere architecture
# ## hence using attention=dot_product
# # Then run training

!python3 -m maxtext.trainers.pre_train.train \
  /content/maxtext/src/maxtext/configs/base.yml \
  model_name=qwen3-0.6b \
  hardware=tpu \
  dataset_type=synthetic \
  reuse_example_batch=1 \
  per_device_batch_size=1 \
  max_target_length=512 \
  base_output_directory=/tmp/maxtext_runs \
  run_name=qwen3_scaled_tpu \
  enable_checkpointing=False \
  skip_jax_distributed_system=True \
  attention=dot_product \
  steps=50 \
  mu_dtype=bfloat16 \
  2>&1 | tee /tmp/training_log_tpu_0.6b.txt

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-06-18 10:45:27.321426: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-18 10:45:27.360828: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructi